# pyCisTopic scATAC processing

Code based on https://pycistopic.readthedocs.io/en/latest/notebooks/human_cerebellum.html

- here, I am calling all peaks without any thresholding to use this callset for IDR peakset creation

In [1]:
import pycisTopic
import pandas as pd
import os
pycisTopic.__version__

'2.0a0'

In [ ]:
!ls /atac/COLO/bamfile_filtered

In [ ]:
!ls /atac/COLO/fragments/

In [ ]:
!pwd

In [ ]:
os.chdir(os.getcwd())

In [ ]:
out_dir = "outs_for_idr"
os.makedirs(out_dir, exist_ok = True)

In [ ]:
import glob

def load_fragment_file(filepath, chunksize=100_000):
    chunks = []
    for chunk in pd.read_csv(
        filepath,
        sep="\t",
        header=None,
        names=["chrom", "start", "end", "barcode", "count"],
        comment="#",
        usecols=["barcode"],
        dtype={"barcode": "category"},
        chunksize=chunksize
    ):
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)

def load_fragments_from_directory(directory, pattern="*.tsv.gz"):
    files = glob.glob(os.path.join(directory, pattern))
    
    if not files:
        print(f"No files found matching '{pattern}' in {directory}")
        return pd.DataFrame()
    
    all_dfs = []
    for filepath in files:
        sample_name = os.path.basename(filepath).replace("_corrected.tsv.gz", "")
        print(f"Processing: {sample_name}...")
        df = load_fragment_file(filepath)
        df["barcode"] = df["barcode"].astype(str) + "-" + sample_name
        df["sample"] = sample_name
        all_dfs.append(df)
        print(f"  → {len(df):,} rows loaded")

    final_df = pd.concat(all_dfs, ignore_index=True)
    barcodes_df = final_df.drop_duplicates(subset="barcode").reset_index(drop=True)
    print(f"\nDone. {len(barcodes_df):,} unique barcodes across {len(files)} files.")
    return barcodes_df


barcodes_df = load_fragments_from_directory(
    directory="/atac/COLO/fragments/",
    pattern="*_corrected.tsv.gz"
)
barcodes_df

In [ ]:
barcodes_df['sample'].value_counts()

# Get pseudobulks

In [ ]:
fragments_dict = {
    "COLO829BL":"fragments/COLO829BL_corrected.tsv.gz",
    "COLO829_rep1": "fragments/COLO829_rep1_corrected.tsv.gz",
    "COLO829_rep2": "fragments/COLO829_rep2_corrected.tsv.gz"
}

In [ ]:
chromsizes = pd.read_table(
    "http://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes",
    header = None,
    names = ["Chromosome", "End"]
)
chromsizes.insert(1, "Start", 0)
chromsizes.head()

In [ ]:
barcodes_df = barcodes_df.set_index("barcode")

In [ ]:
barcodes_df

In [ ]:
from pycisTopic.pseudobulk_peak_calling import export_pseudobulk
os.makedirs(os.path.join(out_dir, "consensus_peak_calling"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bed_files"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bw_files"), exist_ok = True)


bw_paths, bed_paths = export_pseudobulk(
    input_data = barcodes_df,
    variable = "sample",
    sample_id_col = "sample",
    chromsizes = chromsizes,
    bed_path = os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bed_files"),
    bigwig_path = os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bw_files"),
    path_to_fragments = fragments_dict,
    n_cpu = 15,
    normalize_bigwig = True,
    temp_dir = "/atac/tmp",
    split_pattern = "-"
)

In [ ]:
with open(os.path.join(out_dir, "consensus_peak_calling/bw_paths.tsv"), "wt") as f:
    for v in bw_paths:
        _ = f.write(f"{v}\t{bw_paths[v]}\n")

In [ ]:
with open(os.path.join(out_dir, "consensus_peak_calling/bed_paths.tsv"), "wt") as f:
    for v in bed_paths:
        _ = f.write(f"{v}\t{bed_paths[v]}\n")

# Inferring the consensus peaks

In [ ]:
bw_paths = {}
with open(os.path.join(out_dir, "consensus_peak_calling/bw_paths.tsv")) as f:
    for line in f:
        v, p = line.strip().split("\t")
        bw_paths.update({v: p})

In [ ]:
bed_paths = {}
with open(os.path.join(out_dir, "consensus_peak_calling/bed_paths.tsv")) as f:
    for line in f:
        v, p = line.strip().split("\t")
        bed_paths.update({v: p})

In [ ]:
bed_paths

In [ ]:
# keep all peaks

In [ ]:
from pycisTopic.pseudobulk_peak_calling import peak_calling
macs_path = "macs2"

os.makedirs(os.path.join(out_dir, "consensus_peak_calling/MACS"), exist_ok = True)

narrow_peak_dict = peak_calling(
    macs_path = macs_path,
    bed_paths = bed_paths,
    outdir = os.path.join(os.path.join(out_dir, "consensus_peak_calling/MACS")),
    genome_size = 'hs',
    n_cpu = 15,
    input_format = 'BEDPE',
    shift = 73,
    ext_size = 146,
    keep_dup = 'all',
    q_value = 1,
    _temp_dir = '/tmp/ray_spill'
)

In [ ]:
pip list